# Silver Preprocessing — Central Bank News (BOE / ECB / FED)

Converts raw JSONL documents from `data/raw/news/{boe,ecb,fed}` into the
project Silver-layer macro news schema.

**Silver schema**:

| Column | Type | Notes |
|---|---|---|
| `timestamp_utc` | datetime64[ns, UTC] | Parsed from `timestamp_published` |
| `article_id` | str | SHA-256 of URL (fallback: title+date) |
| `source` | str | `boe` / `ecb` / `fed` |
| `document_type` | str | `speech` / `statement` / `summary` / `policy` / `press_release` / `bulletin` / `regulation` / `other` |
| `title` | str | Cleaned headline |
| `content` | str | Cleaned full text |
| `speaker` | str | Speaker name or null |
| `url` | str | Original URL |
| `language` | str | `en` / other (from ECB metadata) |

Output:
- `data/processed/macro/news/boe/year={YYYY}/month={MM}/news_cleaned.parquet`
- `data/processed/macro/news/ecb/year={YYYY}/month={MM}/news_cleaned.parquet`
- `data/processed/macro/news/fed/year={YYYY}/month={MM}/news_cleaned.parquet`


In [1]:
from __future__ import annotations

import hashlib
import json
import re
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("..")
RAW_NEWS_DIR = ROOT / "data" / "raw" / "news"
SILVER_DIR   = ROOT / "data" / "processed" / "macro" / "news"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw news dir  : {RAW_NEWS_DIR.resolve()}")
print(f"Silver output : {SILVER_DIR.resolve()}")


Raw news dir  : C:\Users\LENOVO\Desktop\PI Project\FX-AlphaLab\data\raw\news
Silver output : C:\Users\LENOVO\Desktop\PI Project\FX-AlphaLab\data\processed\macro\news


In [2]:
SILVER_COLUMNS = [
    "timestamp_utc",
    "article_id",
    "source",
    "document_type",
    "title",
    "content",
    "speaker",
    "url",
    "language",
]

# Canonical document type mapping per source
DOC_TYPE_MAP: dict[str, dict[str, str]] = {
    "boe": {
        "speeches":   "speech",
        "statements": "statement",
        "summaries":  "summary",
    },
    "ecb": {
        "speeches":      "speech",
        "policy":        "policy",
        "pressreleases": "press_release",
        "bulletins":     "bulletin",
    },
    "fed": {
        "speeches":   "speech",
        "policy":     "policy",
        "regulation": "regulation",
        "other":      "other",
    },
}

# Normalize known raw labels into canonical Silver document types
RAW_DOC_TYPE_ALIASES: dict[str, str] = {
    "speech":                   "speech",
    "speeches":                 "speech",
    "boe_speech":               "speech",
    "statement":                "statement",
    "statements":               "statement",
    "monetary_policy_summary":  "summary",
    "summary":                  "summary",
    "summaries":                "summary",
    "policy":                   "policy",
    "policy_decision":          "policy",
    "press_release":            "press_release",
    "pressrelease":             "press_release",
    "pressreleases":            "press_release",
    "bulletin":                 "bulletin",
    "bulletins":                "bulletin",
    "regulation":               "regulation",
    "regulations":              "regulation",
    "other":                    "other",
}


In [3]:
def clean_text(text: str | None) -> str:
    """Strip HTML, fix line endings, collapse whitespace."""
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def normalize_document_type(raw_doc_type: str, default_doc_type: str) -> str:
    """Map raw record label to canonical Silver document_type.
    Falls back to default_doc_type (from DOC_TYPE_MAP) when not in aliases.
    """
    candidate = re.sub(r"[\s-]+", "_", (raw_doc_type or "").strip().lower())
    if candidate in RAW_DOC_TYPE_ALIASES:
        return RAW_DOC_TYPE_ALIASES[candidate]

    fallback = re.sub(r"[\s-]+", "_", (default_doc_type or "other").strip().lower())
    return RAW_DOC_TYPE_ALIASES.get(fallback, fallback)  # return fallback itself if not in aliases


def make_article_id(url: str | None, title: str | None, date: str | None) -> str:
    """SHA-256 of URL; fallback to title+date if URL is missing or relative."""
    raw = url or ""
    if not raw or raw.startswith("/"):
        raw = f"{title}|{date}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


def parse_timestamp(ts: str | None) -> pd.Timestamp | pd.NaT:
    """Parse ISO timestamp and coerce to UTC."""
    if not ts:
        return pd.NaT
    try:
        t = pd.Timestamp(ts)
        if t.tzinfo is None:
            t = t.tz_localize("UTC")
        else:
            t = t.tz_convert("UTC")
        return t
    except Exception:
        return pd.NaT


def load_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return records


In [4]:
def process_source(source: str) -> pd.DataFrame:
    """Load all JSONL files for a source, normalise to Silver schema."""
    source_dir = RAW_NEWS_DIR / source
    if not source_dir.exists():
        print(f"  [SKIP] {source} directory not found")
        return pd.DataFrame(columns=SILVER_COLUMNS)

    rows: list[dict] = []
    file_type_map = DOC_TYPE_MAP.get(source, {})

    for jsonl_file in sorted(source_dir.glob("*.jsonl")):
        stem_parts = jsonl_file.stem.split("_")
        file_key = stem_parts[0]
        doc_type_default = file_type_map.get(file_key, "other")

        records = load_jsonl(jsonl_file)
        print(f"  {jsonl_file.name}: {len(records):,} records  [{doc_type_default}]")

        for rec in records:
            ts = parse_timestamp(rec.get("timestamp_published"))
            if ts is pd.NaT:
                continue

            url      = str(rec.get("url") or "")
            title    = clean_text(rec.get("title"))
            date_str = str(ts.date())
            raw_type = str(rec.get("document_type") or file_key)
            doc_type = normalize_document_type(raw_type, doc_type_default)

            rows.append({
                "timestamp_utc":  ts,
                "article_id":     make_article_id(url, title, date_str),
                "source":         source,
                "document_type":  doc_type,
                "title":          title,
                "content":        clean_text(rec.get("content")),
                "speaker":        str(rec.get("speaker") or "") or None,
                "url":            url or None,
                "language":       str(rec.get("language") or "en"),
            })

    if not rows:
        return pd.DataFrame(columns=SILVER_COLUMNS)

    df = pd.DataFrame(rows, columns=SILVER_COLUMNS)
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
    return df.sort_values("timestamp_utc").reset_index(drop=True)


In [5]:
print("=== BOE ===")
boe_df = process_source("boe")
print(f"  Total: {len(boe_df):,} rows | {boe_df['timestamp_utc'].min()} → {boe_df['timestamp_utc'].max()}")

print("\n=== ECB ===")
ecb_df = process_source("ecb")
print(f"  Total: {len(ecb_df):,} rows | {ecb_df['timestamp_utc'].min()} → {ecb_df['timestamp_utc'].max()}")

print("\n=== FED ===")
fed_df = process_source("fed")
print(f"  Total: {len(fed_df):,} rows | {fed_df['timestamp_utc'].min()} → {fed_df['timestamp_utc'].max()}")


=== BOE ===
  speeches_20260411.jsonl: 1,427 records  [speech]
  statements_20260411.jsonl: 2,984 records  [statement]
  summaries_20260411.jsonl: 90 records  [summary]
  Total: 4,501 rows | 2000-01-03 00:00:00+00:00 → 2025-12-23 00:00:00+00:00

=== ECB ===
  bulletins_20260413.jsonl: 5 records  [bulletin]
  policy_20260413.jsonl: 1,066 records  [policy]
  pressreleases_20260413.jsonl: 1,478 records  [press_release]
  speeches_20260413.jsonl: 2,397 records  [speech]
  Total: 4,946 rows | 2000-01-03 00:00:00+00:00 → 2026-04-12 21:03:16.224915+00:00

=== FED ===
  other_20260411.jsonl: 894 records  [other]
  policy_20260411.jsonl: 1,094 records  [policy]
  regulation_20260411.jsonl: 3,567 records  [regulation]
  speeches_20260411.jsonl: 2,025 records  [speech]
  Total: 7,580 rows | 2000-01-04 00:00:00+00:00 → 2025-12-30 00:00:00+00:00


In [6]:
all_df = pd.concat([boe_df, ecb_df, fed_df], ignore_index=True)

print(f"Total records : {len(all_df):,}")
print()

# Missing values
missing = all_df[["timestamp_utc", "title", "content", "source", "document_type"]].isna().sum()
print("Missing values:")
print(missing.to_string())
print()

# Empty content
empty_content = (all_df["content"].str.strip() == "").sum()
empty_pct = (100 * empty_content / len(all_df)) if len(all_df) else 0.0
print(f"Empty content : {empty_content:,} ({empty_pct:.1f}%)")

# Duplicate article IDs
dup_ids = all_df["article_id"].duplicated().sum()
print(f"Duplicate IDs : {dup_ids:,}")

# Distribution by source + document_type
print()
print("Records by source / document_type:")
display(
    all_df.groupby(["source", "document_type"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["source", "count"], ascending=[True, False])
)

# Date range by source
print("\nDate range by source:")
display(
    all_df.groupby("source")["timestamp_utc"]
    .agg(first="min", last="max", count="count")
    .reset_index()
)


Total records : 17,027

Missing values:
timestamp_utc    0
title            0
content          0
source           0
document_type    0

Empty content : 0 (0.0%)
Duplicate IDs : 315

Records by source / document_type:


,source,document_type,count
0,boe,press_release,2984
1,boe,speech,1427
2,boe,summary,90
6,ecb,speech,2397
5,ecb,press_release,1478
4,ecb,policy,1066
3,ecb,bulletin,5
9,fed,regulation,3567
10,fed,speech,2025
8,fed,policy,1094



Date range by source:


,source,first,last,count
0,boe,2000-01-03 00:00:00+00:00,2025-12-23 00:00:00+00:00,4501
1,ecb,2000-01-03 00:00:00+00:00,2026-04-12 21:03:16.224915+00:00,4946
2,fed,2000-01-04 00:00:00+00:00,2025-12-30 00:00:00+00:00,7580


In [7]:
before = len(all_df)

# Drop duplicate article IDs (keep first occurrence)
all_df = all_df.drop_duplicates(subset="article_id", keep="first")

# Drop rows with empty content AND empty title (truly empty documents)
all_df = all_df[~((all_df["content"].str.strip() == "") & (all_df["title"].str.strip() == ""))]

after = len(all_df)
print(f"Rows before dedup/empty drop : {before:,}")
print(f"Rows after                   : {after:,}")
print(f"Removed                      : {before - after:,}")


Rows before dedup/empty drop : 17,027
Rows after                   : 16,712
Removed                      : 315


In [8]:
def export_silver_partitioned(df: pd.DataFrame) -> None:
    """Write to data/processed/macro/news/{source}/year={y}/month={m}/news_cleaned.parquet"""
    if df.empty:
        print("Nothing to export.")
        return

    df = df.copy()
    df["year"]  = df["timestamp_utc"].dt.year
    df["month"] = df["timestamp_utc"].dt.month

    groups = df.groupby(["source", "year", "month"])
    total_files = 0

    for (source, year, month), group in groups:
        part_dir = SILVER_DIR / source / f"year={year}" / f"month={month:02d}"
        part_dir.mkdir(parents=True, exist_ok=True)
        out_path = part_dir / "news_cleaned.parquet"

        group_out = group.drop(columns=["year", "month"])
        group_out.to_parquet(out_path, index=False, engine="pyarrow")
        total_files += 1

    print(f"Exported {len(df):,} rows across {total_files} partition files.")
    print(f"Output root: {SILVER_DIR.resolve()}")


export_silver_partitioned(all_df)


Exported 16,712 rows across 919 partition files.
Output root: C:\Users\LENOVO\Desktop\PI Project\FX-AlphaLab\data\processed\macro\news


In [9]:
# Spot-check: read back a sample partition and verify schema
parquet_files = sorted(SILVER_DIR.rglob("news_cleaned.parquet"))
print(f"Total parquet partitions written: {len(parquet_files)}")

if parquet_files:
    sample = pd.read_parquet(parquet_files[0])
    print(f"\nSample partition: {parquet_files[0]}")
    print(f"Shape: {sample.shape}")
    print(f"Columns: {list(sample.columns)}")
    print()
    display(sample.head(3))

    # Schema compliance check
    missing_cols = [c for c in SILVER_COLUMNS if c not in sample.columns]
    extra_cols   = [c for c in sample.columns if c not in SILVER_COLUMNS]
    print(f"Missing columns : {missing_cols}")
    print(f"Extra columns   : {extra_cols}")
    print(f"Schema OK       : {len(missing_cols) == 0}")
else:
    print("No parquet files found. Run export cell first.")


Total parquet partitions written: 919

Sample partition: ..\data\processed\macro\news\boe\year=2000\month=01\news_cleaned.parquet
Shape: (14, 9)
Columns: ['timestamp_utc', 'article_id', 'source', 'document_type', 'title', 'content', 'speaker', 'url', 'language']



,timestamp_utc,article_id,source,document_type,title,content,speaker,url,language
0,2000-01-03 00:00:00+00:00,ce02a335ab01d1cc,boe,press_release,Year 2000 - So Far So Good,Very few Y2K problems have been reported and t...,None,https://www.bankofengland.co.uk/news/2000/janu...,en
1,2000-01-04 00:00:00+00:00,07a2c26748ad86b9,boe,press_release,Bank of England Euro Bills,Published on \n 04 January 2000\nAn additional...,None,https://www.bankofengland.co.uk/news/2000/janu...,en
2,2000-01-07 00:00:00+00:00,082e910ef3dbd3c7,boe,speech,Monetary Policy: Theory in Practice - address ...,Published on \n 07 January 2000\nMonetary Poli...,Alan Taylor,https://www.bankofengland.co.uk/speech/2000/mo...,en


Missing columns : []
Extra columns   : []
Schema OK       : True


In [10]:
print("=== Silver Preprocessing Complete ===")
print()
print(f"Sources processed : BOE, ECB, FED")
print(f"Total documents   : {len(all_df):,}")
print()
print("Columns at Silver layer:")
for col in SILVER_COLUMNS:
    null_count = all_df[col].isna().sum()
    print(f"  {col:<20} nulls={null_count:>6,}")
print()
print("Ready for macro agent (Module C)")


=== Silver Preprocessing Complete ===

Sources processed : BOE, ECB, FED
Total documents   : 16,712

Columns at Silver layer:
  timestamp_utc        nulls=     0
  article_id           nulls=     0
  source               nulls=     0
  document_type        nulls=     0
  title                nulls=     0
  content              nulls=     0
  speaker              nulls=12,116
  url                  nulls=     0
  language             nulls=     0

Ready for macro agent (Module C)
